# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 137.37it/s]


2026-11-03 14:43:46.053 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:902 - Data batch-empirical estimation of propensity score.


2026-11-03 14:43:46.061 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:951 - Data prediction of expected reward based on gbm model.


2026-11-03 14:43:47.349 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1068 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


... (2,265 outputs left out)

2026-11-03 14:44:23.916 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-11-03 14:44:24.338 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.496952,0.462656,0.530875,0.017156,b-ipw,reward_0
1,0.500313,0.499298,0.501303,0.000511,dm,reward_0
2,0.495782,0.463439,0.528189,0.016553,dr,reward_0
3,0.500313,0.499298,0.501303,0.000515,dros-opt,reward_0
4,0.495782,0.463236,0.528091,0.016551,dros-pess,reward_0
5,0.496164,0.462496,0.528971,0.017109,ipw,reward_0
6,0.495970,0.463710,0.529917,0.016950,rep,reward_0
7,0.495785,0.463906,0.528491,0.016429,sndr,reward_0
8,0.495848,0.462488,0.530261,0.017182,snips,reward_0
9,0.495782,0.463320,0.529173,0.016595,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 270.35it/s]


2026-11-03 14:44:24.905 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1300 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<13:03,  1.28it/s]

SVI:   0%|          | 1/1000 [00:00<13:03,  1.28it/s, loss=2752.2305]

... (4,378 outputs left out)

2026-11-03 14:45:41.412 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-11-03 14:45:41.838 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1186 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.500277,0.468675,0.530811,0.015940,b-ipw,reward_0
1,0.500070,0.499062,0.501134,0.000531,dm,reward_0
2,0.499627,0.469481,0.530564,0.015679,dr,reward_0
3,0.500070,0.499022,0.501093,0.000534,dros-opt,reward_0
4,0.499627,0.468393,0.530357,0.015832,dros-pess,reward_0
5,0.497931,0.466976,0.528651,0.015837,ipw,reward_0
6,0.499799,0.469071,0.530565,0.015699,rep,reward_0
7,0.499625,0.468476,0.531445,0.015804,sndr,reward_0
8,0.499647,0.469249,0.530621,0.015748,snips,reward_0
9,0.499627,0.469412,0.531238,0.015776,sg-dr,reward_0
